In [1]:
import torch
from torch.utils.data import DataLoader
from PIL import Image
from torchvision import models
from tqdm import tqdm
import os
import sys

In [2]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, split, preprocess, model):
        self.split = split
        self.preprocess = preprocess
        self.model = model

        with open(f'./data/data/{split}.csv', 'r') as f:
            lines = f.readlines()
        self.filename, self.y, self.a = [], [], []
        for line in lines[1:]:
            _, filename, y = line.rstrip().split(',')
            self.filename.append(filename)
            self.y.append(int(y))
        
        self.imgs = [self.preprocess(Image.open(f'./data/data/{self.split}/{filename}').convert('RGB')).cuda() for filename in self.filename]

        self.features = []
        with torch.no_grad():
            for i in tqdm(range(0, len(self.imgs), 16)):
                if i + 16 > len(self.imgs):
                    break
                self.features.append(self.model(torch.stack(self.imgs[i:i+16])))

        self.features = torch.cat(self.features, dim=0)
        self.y = self.y[:self.features.shape[0]]
        print(self.features.shape)

    def __getitem__(self, sample_n):
        return self.features[sample_n], self.y[sample_n]
    
    def __len__(self):
        return len(self.y)

In [3]:
weights = models.ResNet50_Weights.IMAGENET1K_V1
resnet   = models.resnet50(weights=weights).eval()
resnet.fc = torch.nn.Identity()
preprocess = weights.transforms()
resnet.eval()
resnet.cuda()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
datasets = {split: Dataset(split, preprocess, resnet) for split in ['train', 'val']}
train_loader = DataLoader(datasets['train'], batch_size = 16, shuffle = True)
val_loader = DataLoader(datasets['val'], batch_size = 16, shuffle = False)

In [15]:
from torch import nn
p=0.7
model = nn.Sequential(
    nn.Dropout(p),
    nn.Linear(2048, 1)
)
model.cuda()

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:

        for images, labels in loader:
            images = images.cuda()
            labels = labels.float().cuda()

            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = (torch.sigmoid(logits) > 0.5).long()

            total_loss += loss.item() * images.size(0)
            total_correct += (preds == labels.long()).sum().item()
            total_samples += images.size(0)

        if not is_train:
            print(f"Loss: {total_loss / total_samples} | Acc: {total_correct / total_samples}")

    return (
        total_loss / total_samples,
        total_correct / total_samples,
    )

EPOCHS = 20
for i in range(EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, True)
    val_loss, val_acc = run_epoch(model, val_loader, False)

Loss: 0.5216424473234125 | Acc: 0.7787162162162162
Loss: 0.48831260808416316 | Acc: 0.7787162162162162
Loss: 0.47254660846413793 | Acc: 0.7787162162162162
Loss: 0.4591795322862831 | Acc: 0.7795608108108109
Loss: 0.4479163627366762 | Acc: 0.7804054054054054
Loss: 0.43885194852545456 | Acc: 0.7905405405405406
Loss: 0.43138418568147197 | Acc: 0.8074324324324325
Loss: 0.42424376429738225 | Acc: 0.8158783783783784
Loss: 0.4202835410833359 | Acc: 0.8268581081081081
Loss: 0.41533584731656153 | Acc: 0.831081081081081
Loss: 0.4121816579151798 | Acc: 0.8353040540540541
Loss: 0.4098125674031876 | Acc: 0.8378378378378378
Loss: 0.40888700855744853 | Acc: 0.8293918918918919
Loss: 0.40729765312091726 | Acc: 0.831081081081081
Loss: 0.40590344027087494 | Acc: 0.8327702702702703
Loss: 0.4061913667498408 | Acc: 0.8268581081081081
Loss: 0.40519392732027415 | Acc: 0.8277027027027027
Loss: 0.407228755185733 | Acc: 0.8209459459459459
Loss: 0.40673034255569046 | Acc: 0.8192567567567568
Loss: 0.406004992691246